In [0]:
from  pyspark.sql.functions import col, lower, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# Acessando a tabela Delta Bronze

df_bronze = spark.table(
    "projeto_meio_ambiente.datambiente_2015_2021.bronze_datambiente_15_21"
)

In [0]:
#Visuzalização da Estruturas

df_bronze.printSchema()

In [0]:
#Selecionando colunas

df_silver = df_bronze.select(
    "_c0",
    "ID",
    "Data",
    "Hora",
    "Estacao",
    "Codigo",
    "Poluente",
    "Valor",
    "Unidade",
    "Tipo"
)
df_silver.show(5)
df_silver.printSchema()

In [0]:
#Removendo linhas com valores nulos

df_silver = df_silver.dropna(
    subset=["_c0", "ID", "Data", "Hora", "Estacao", "Codigo", "Poluente", "Valor", "Unidade", "Tipo"]
)
df_silver.show(5,truncate=False)

In [0]:
#padronização de colunas

df_silver = df_silver.withColumn(
    "Data", 
    lower(col("data"))
)

df_silver = df_silver.withColumn(
    "Hora", 
    lower(col("hora"))
)

df_silver = df_silver.withColumn(
    "Estacao", 
    lower(col("estacao"))
)

df_silver = df_silver.withColumn(
    "Codigo", 
    lower(col("codigo"))
)

df_silver = df_silver.withColumn(
    "Poluente", 
    lower(col("poluente"))
)

df_bronze = df_silver.withColumn(
    "Valor", 
    lower(col("valor"))
)

df_bronze = df_silver.withColumn(
    "Unidade", 
    lower(col("unidade"))
)

df_silver = df_silver.withColumn(
    "Tipo", 
    lower(col("tipo"))
)

df_silver.show(10)

In [0]:
df_silver = df_silver.drop('_c0','Hora', 'Codigo')
df_silver.show()

In [0]:
df_silver = df_silver.withColumn(
    "Mes",
    month(col("Data"))
)
df_silver.show()

In [0]:
#criando a coluna ano

df_silver = df_silver.withColumn(
    "Ano",
    year(col("Data"))
)

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "Nome_mês",
    when(col("Mes") == 1, "Janeiro")
    .when(col("Mes") == 2, "Fevereiro")
    .when(col("Mes") == 3, "Março")
    .when(col("Mes") == 4, "Abril")
    .when(col("Mes") == 5, "Maio")
    .when(col("Mes") == 6, "Junho")
    .when(col("Mes") == 7, "Julho")
    .when(col("Mes") == 8, "Agosto")
    .when(col("Mes") == 9, "Setembro")
    .when(col("Mes") == 10, "Outubro")
    .when(col("Mes") == 11, "Novembro")
    .otherwise("Dezembro")
)

df_silver.show(5)

In [0]:
df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("projeto_meio_ambiente.datambiente_2015_2021.silver_datambiente_15_21")

In [0]:
df = spark.table("projeto_meio_ambiente.datambiente_2015_2021.silver_datambiente_15_21")
df.show()

In [0]:
%sql
select * FROM projeto_meio_ambiente.datambiente_2015_2021.silver_datambiente_15_21